# 20 — Pretrigger noise and trace-specific window review

Use FLOVOpy's `detect_pretrigger_window()` through the local batch/export adapter. It is also discoverable in `flovopy.core.vdap`. The initial five seconds are a working noise assumption, screened for invalid reference data. Arrival evidence does **not** recover the acquisition trigger, prove a ten-second buffer or repair recorded clock errors.

Candidates are deliberately separate from accepted windows. No event end is automatically assigned at record end. For an emergent rockfall/PDC signal, onset support describes detector evidence rather than a uniquely known physical start.

**Manual review is optional.** Set `WINDOW_POLICY='fixed'` or `'detected'` in notebook 30 and Run All there. Fixed mode does not need this detection notebook. Automatic selections are saved separately from manual review decisions.


## Choose a larger batch

Edit `FILES_PER_MONTH` below: 50 gives up to 200 files across the four months. Unlike the old pilot, this samples files evenly across each month's sorted list and can include several recordings per day. Change `SAMPLE_MODE` to `'all'` to use the entire WAV archive; `MONTHS` is ignored in that mode. `MAX_FILES` optionally caps the batch using a spread-out subset.

The selection is written to the shared configuration, with a backup, so notebooks 00/30/40/50 use the same batch when rerun. Changed inputs/settings create a separate run; previous reviews are preserved. A previously frozen publication selection remains pinned to its old run until explicitly revised. Sampling many files is not equivalent to sampling independent earthquakes: a file may contain several signals or overlap another file.

The batch caches each file and keeps arrays only for a few spread-out plot examples. Reruns reuse completed caches (example files are reloaded for plotting). Optional frequency/Hilbert calculations are off for the large batch and shown separately for one example; they do not change onset selection.

In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), Path.cwd() / '20_analysis', *Path.cwd().parents]
ANALYSIS_DIR = next((p for p in _candidates if (p / 'analysis_workflow.py').is_file()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('Start in 20_analysis or its project root, beside analysis_workflow.py.')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))
from analysis_workflow import load_context, describe_context
from IPython.display import display, Image, Markdown
import matplotlib.pyplot as plt
import pandas as pd

# Choose the batch here. These settings are saved for the other analysis notebooks.
SAMPLE_MODE = None    # None: keep current shared selection; 'sample' or 'all': explicitly change it
FILES_PER_MONTH = 50     # 50 × 4 months = 200 files; raise to 100 or 250 for larger samples
MONTHS = ('1991/05', '1991/06', '1991/07', '1991/08')  # used for sample/pilot modes
MAX_FILES = None         # optional spread-out overall cap; None leaves the selection uncapped
SAVE_SELECTION = True
from analysis_workflow import configure_selection
if SAVE_SELECTION and SAMPLE_MODE is not None:
    configure_selection(ANALYSIS_DIR, mode=SAMPLE_MODE, files_per_month=FILES_PER_MONTH,
                        months=MONTHS, max_files=MAX_FILES)
ctx = load_context(ANALYSIS_DIR)
display(describe_context(ctx))
print('Selected event files:', len(ctx['files']))


In [ ]:
from flovopy.processing.detection import PretriggerSettings, detect_pretrigger_window
from analysis_workflow import run_window_proposals
COMPUTE_BATCH_FREQUENCY = False
COMPUTE_BATCH_HILBERT = False
FORCE_RECOMPUTE = False
MAX_EXAMPLE_FILES = 4
settings = PretriggerSettings(minimum_noise_s=5, reference_end_s=5,
    noise_guard_s=ctx['config']['processing']['noise_guard_s'], network_min_stations=2,
    compute_frequency=COMPUTE_BATCH_FREQUENCY, compute_hilbert=COMPUTE_BATCH_HILBERT)
result, review = run_window_proposals(ctx, settings, force=FORCE_RECOMPUTE,
                                    max_detail_files=MAX_EXAMPLE_FILES)
print('Selected files:', len(ctx['files']), '; analysed files:', result['qc'].record_id.nunique() if len(result['qc']) else 0,
      '; analysed traces:', len(result['qc']))
display(result['files'].groupby('status', dropna=False).size().rename('files').to_frame())
display(review.reindex(columns=['trace_key', 'trace_id', 'selected_onset_file_s', 'selected_onset_method',
    'onset_support_start_file_s', 'onset_support_end_file_s', 'proposed_noise_end_file_s',
    'noise_proposal_status', 'review_status', 'qc_flags']).head(30))


## Noise-window lengths through time

The upper panel shows per-trace **proposed noise duration**, not simply its file-relative end time. Grey × symbols mark five-second minimum-only fallbacks; these are assumptions, not detections. The middle panel gives equal-station medians and station spread for each analysed file, plus the earliest selected onset from file start. A multi-component station gets one vote. Invalid windows stay missing and their counts remain visible below.

The one-second guard means a noise window generally ends before the detected rise; earlier transients can shorten it further. Neither series is a direct measurement of the MDETECT trigger/buffer setting. The 5/10-second lines are visual references. Dates are recorded file times, with no clock correction.

This configuration assumes the first five seconds are noise. It cannot establish quiet prefixes shorter than five seconds; early/ambiguous records require inspection rather than treating the fallback as a measured minimum.


In [ ]:
from analysis_workflow import plot_noise_windows_through_time
if len(result['qc']):
    fig, trace_windows, event_windows = plot_noise_windows_through_time(ctx, result['qc'], review)
    display(Image(filename=str(ctx['out'] / 'figures/noise_windows_through_time.png')))
    plt.close(fig)
    display(event_windows.head(20))
    display(trace_windows.groupby('proposal_class').size().rename('traces').to_frame())
else:
    print('No successful trace analyses. Inspect the file-status table above.')
print('CSV tables and PNG/PDF/SVG:', ctx['out'])


## Full records and onset zooms

A dot is a proposal and a short dashed line is a proposed noise end. First support bounds are in the table. Weak channels keep only the assumed initial minimum; contradictory guards remain unresolved. Inspect all channels, early transients, padding and later secondary activity.

In [ ]:
from dataclasses import replace
from pretrigger_qc import plot_record, plot_frequency_record, plot_hilbert_trace
for record_id, records in list(result['details'].items())[:MAX_EXAMPLE_FILES]:
    full_seconds = max(r['file_offset_s'] + r['qc']['duration_s'] for r in records)
    for suffix, plot_settings in [('full', replace(settings, plot_seconds=full_seconds)), ('zoom', settings)]:
        stem = ctx['out'] / 'figures' / f'windows_{record_id}_{suffix}'
        fig = plot_record(records, plot_settings, stem, max_channels=100)
        display(Image(filename=str(stem.with_suffix('.png'))))
        plt.close(fig)


## Independent frequency diagnostics

Band-limited PSD change and signed Hilbert instantaneous frequency can reveal a transition without a large amplitude jump. Neither absence of a frequency shift nor agreement between descriptors establishes quiet noise. Hilbert estimates can be negative or unstable for broadband signals; coverage matters. These diagnostics do not automatically accept or move a window.

In [ ]:
# Detailed frequency diagnostics for ONE retained example, independent of batch switches.
from pretrigger_qc import analyse_files
from dataclasses import replace
if result['details']:
    records = next(iter(result['details'].values()))
    example_file = records[0]['qc']['file']
    diagnostic_settings = replace(settings, compute_frequency=True, compute_hilbert=True)
    diagnostic = analyse_files([example_file], ctx['out'] / 'frequency_example', diagnostic_settings)
    diagnostic_records = next(iter(diagnostic['details'].values()))
    for function, suffix in [(plot_frequency_record, 'frequency'), (plot_hilbert_trace, 'hilbert')]:
        stem = ctx['out'] / 'figures' / f'window_diagnostic_{suffix}'
        fig = function(diagnostic_records if suffix == 'frequency' else diagnostic_records[0], diagnostic_settings, stem)
        display(Image(filename=str(stem.with_suffix('.png'))))
        plt.close(fig)


## Selected policy windows and optional manual overrides

Fixed and detected policies require no individual acceptance. Notebook 30's switch selects the active policy. The table below shows the actual windows used for metrics. Automatic selections do not overwrite manual review decisions. Advanced manual mode is `reviewed`; only that mode uses `REVIEW_UPDATES` / accepted bounds. Record-end windows are data windows, not measured coda durations.


In [ ]:
from analysis_workflow import save_review_decisions, select_analysis_windows, window_policy
REVIEW_UPDATES = {}  # optional explicit human decisions for the advanced reviewed policy
if REVIEW_UPDATES:
    save_review_decisions(ctx, REVIEW_UPDATES)
selected, issues = select_analysis_windows(ctx)
print('Active policy:', window_policy(ctx))
display(selected.head(30))
display(issues.head(30))
print('Selected windows:', len(selected))
print('Policy windows:', ctx['out'] / 'selected_windows.csv')


Continue to notebook **30**, choose fixed or detected, then Run All. `selected_windows.csv` records actual automatic-policy intervals; `window_review.csv` is optional manual-review data. No per-event manual review is required in fixed/detected modes.
